# `generators.py` Reference

Match simulation: skill-driven (`skilled_match`) and skill-free (`random_match`) baselines.
The only place `Player.skill` is ever read.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

ready


## `score_agent_probability` — logistic skill gap

In [3]:
from tournament.generators import score_agent_probability

print(f'Equal skills (gap=0):  p={score_agent_probability(0.0, 0.0):.3f}')
print(f'Gap +1 (A stronger):   p={score_agent_probability(1.0, 0.0):.3f}')
print(f'Gap +2:                p={score_agent_probability(2.0, 0.0):.3f}')
print(f'Gap -1 (A weaker):     p={score_agent_probability(-1.0, 0.0):.3f}')

Equal skills (gap=0):  p=0.500
Gap +1 (A stronger):   p=0.731
Gap +2:                p=0.881
Gap -1 (A weaker):     p=0.269


## `make_players`

In [4]:
from tournament.generators import make_players

rng = Random(42)
players = make_players(8, rng, skill_sd=1.0)
print('Generated players:')
for p in players:
    print(f'  {p.name}  pid={p.pid}  skill={p.skill:+.3f}')

Generated players:
  P000  pid=0  skill=-0.144
  P001  pid=1  skill=-0.173
  P002  pid=2  skill=-0.111
  P003  pid=3  skill=+0.702
  P004  pid=4  skill=-0.128
  P005  pid=5  skill=-1.497
  P006  pid=6  skill=+0.332
  P007  pid=7  skill=-0.267


## `skilled_match` vs `random_match`

In [5]:
from tournament.generators import skilled_match, random_match

strong = max(players, key=lambda p: p.skill)
weak   = min(players, key=lambda p: p.skill)
print(f'Strong: {strong.name} skill={strong.skill:+.3f}')
print(f'Weak:   {weak.name}   skill={weak.skill:+.3f}')
print(f'P(strong captures agent) = {score_agent_probability(strong.skill, weak.skill):.3f}')

rng_trial = Random(99)
ws = {strong.name: 0, weak.name: 0, 'draw': 0}
wr = {strong.name: 0, weak.name: 0, 'draw': 0}
for _ in range(200):
    for m, d in [(skilled_match(strong, weak, rng_trial), ws),
                 (random_match(strong, weak, rng_trial),  wr)]:
        if m.winner == strong.pid:  d[strong.name] += 1
        elif m.winner == weak.pid:  d[weak.name]   += 1
        else:                       d['draw']       += 1

print(f'\nskilled_match (200 trials): {ws}')
print(f'random_match  (200 trials): {wr}  <- near 50/50')

Strong: P003 skill=+0.702
Weak:   P005   skill=-1.497
P(strong captures agent) = 0.900

skilled_match (200 trials): {'P003': 196, 'P005': 0, 'draw': 4}
random_match  (200 trials): {'P003': 57, 'P005': 38, 'draw': 105}  <- near 50/50


## Single match detail

In [6]:
rng2 = Random(7)
m = skilled_match(strong, weak, rng2)
print(f'winner: {m.winner}  is_draw: {m.is_draw}')
print(f'games:  {[(g.agents_a, g.agents_b) for g in m.games]}')
print(f'bonus_agents_a={m.bonus_agents_a}  bonus_agents_b={m.bonus_agents_b}')
print(f'total_agents_a={m.total_agents_a}  total_agents_b={m.total_agents_b}')

winner: 3  is_draw: False
games:  [(3, 0), (3, 0)]
bonus_agents_a=0  bonus_agents_b=0
total_agents_a=6  total_agents_b=0
